# Create a PyTorch Network

Here we create a neural network using PyTorch.  This tutorial covers several concepts:

* `torch.Tensor` - A *multi-dimensional array* with support for autograd operations like `backward()`.  Also *holds the gradient* w.r.t. the tensor.
* `nn.Module` - Neural network module.  *Convenient way of encapsulating parameters*, with helpers for moving them to GPU, exporting, loading, etc.
* `nn.Parameter` - A kind of Tensor, that is *automatially registered as a parameter when assigned as an attribute* to a `Module`.
* `autograd.Function` - Implements *forward and backward definitions of an autograd operation*.  Every `Tensor` operation creates at least a single `Function` node that connects to function tha created a `Tensor` and *encodes its history*.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Create a network using the forward function

Here we use the `forward` function to create a network.  Note that we only have to define the `forward` function and the backward function (where gradients are computed) will automatically be defined for us using `autograd`.  We can use any of the Tensor operations in the `forward` function.

In [2]:
class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()

        # 1 input channel, 6 output channels, 3x3 square convolution
        # kernel
        self.conv1 = nn.Conv2d(1, 6, 3)
        self.conv2 = nn.Conv2d(6, 16, 3)
        
        # an affine operation: y = Wx + b
        self.fc1 = nn.Linear(16 * 6 * 6, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # Max pooling over a (2, 2) window
        x = F.max_pool2d(F.relu(self.conv1(x)), (2,2))
        # If the size is a square you can only specify a single number
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def num_flat_features(self, x):
        size = x.size()[1:]  # all dimensions except the batch dimension
        num_features = 1
        for s in size:
            num_features *= s
        return num_features


## Return the parameters of the network

The learnable parameters of a model are returned by `net.parameters()`.

In [3]:
net = Net()

print('Created network:\n')
print(net)

params = list(net.parameters())

print('\nPrint number of parameters:\n')
print(len(params))

print('\nSize of parameters:\n')
print(params[0].size())


Created network:

Net(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=576, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

Print number of parameters:

10

Size of parameters:

torch.Size([6, 1, 3, 3])


## Run an input through the network

Here we try a random 32x32 input.  Note: expected input size of this net (LeNet) is 32x32.  To use this net on the MNIST dataset please resize the images from the dataset to 32x32.


In [4]:
input = torch.randn(1, 1, 32, 32)
out = net(input)

print('Network output\n')
print(out)

Network output

tensor([[ 0.1334,  0.0116, -0.0577,  0.0150, -0.1368, -0.0876,  0.0006,  0.1391,
          0.0841, -0.0188]], grad_fn=<AddmmBackward>)


## Set up the network for training

Note that the entire torch.nn package only supports inputs that are a mini-batch of samples, and not a single sample.

For example, `nn.Conv2d` will take in a 4D Tensor of `nSamples x nChannels x Height x Width`.

If you have a single sample, just use `input.unsqueeze(0)` to add a fake batch dimension.

A **loss function** will be defined that tajes the (outputs, target) pair of inputs, and computes a value that estimates how far away the output is from the target.

There are several different **loss functions** under the nn package.  A simple loss is: `nn.MSELoss` which computes the mean-squared error between the input and the target.

In [5]:
# Start by zeroing the gradient buffers of all parameters
# and backprops with random gradients
net.zero_grad()
out.backward(torch.randn(1,10))

output = net(input)
target = torch.randn(10) # a dummy target
target = target.view(1, -1) # make it the same shape as output
criterion = nn.MSELoss()

loss = criterion(output, target)
print(loss)

tensor(1.3896, grad_fn=<MseLossBackward>)


Now, if you follow `loss` in the backward direction, using its `.grad_fn` attribute, you will see a graph of computations that looks like this:

```
input -> conv2d -> relu -> maxpool2d -> conv2d -> relu -> maxpool2d
      -> view -> linear -> relu -> linear -> relu -> linear
      -> MSELoss
      -> loss
```

So, when we call `loss.backward()`, the whole graph is differentiated w.r.t. the loss, and all Tensors in the graph that has `requires_grad=True` will have their `.grad` Tensor accumulated with the gradient.

For illustration:


In [6]:
print(loss.grad_fn) # MSELoss
print(loss.grad_fn.next_functions[0][0]) # Linear
print(loss.grad_fn.next_functions[0][0].next_functions[0][0]) # ReLU


## Backpropagation

To backpropagate the error all we have to do is to `loss.backward()`.  You need to clear the existing gradients though, else gradients will be accumulated to existing gradients.

Now we shall call `loss.backward()`, and have a look at conv1's bias gradients before and after the backard propagation.

In [7]:
net.zero_grad() # zeroes the gradient buffers of all parameters

print('conv1.bias.grad before backward')
print(net.conv1.bias.grad)

loss.backward()

print('conv1.bias.grad after backward')
print(net.conv1.bias.grad)

conv1.bias.grad before backward
tensor([0., 0., 0., 0., 0., 0.])
conv1.bias.grad after backward
tensor([-0.0041, -0.0156,  0.0247,  0.0334, -0.0046, -0.0162])
